In [1]:
import os

In [2]:
file = 'data/test.txt'
path = os.path.join(os.getcwd(), file)
with open(path, 'r') as fp:
    lines = [x.strip() for x in fp.readlines()]

In [3]:
def print_grid(grid, max_r, max_c, highlight=[]):
    for row in range(max_r):
        print(''.join([grid[(row, col)] if (row, col) not in highlight else 'O' for col in range(max_c)]))

In [4]:
def get_adjacent(grid, pos):
    row, col = pos
    adjacent = [(row - 1, col), (row + 1, col), (row, col - 1), (row, col + 1)]
    return {loc for loc in adjacent if grid[loc] != '#'}

In [5]:
def get_direction(old_pos, new_pos):
    old_row, old_col = old_pos
    new_row, new_col = new_pos
    if old_row == new_row:
        return 0 if old_col < new_col else 2
    else:
        return 1 if old_row > new_row else 3

In [6]:
def get_best_paths(grid, scores, start, end):
    score, final_paths, process = scores[start], [start], [start]
    while True:
        if process:
            print(process)
        for pos in process:
            new_process = []
            for loc in get_adjacent(grid, pos):
                if loc == end:
                    final_paths.append(end)
                    return final_paths
                if scores[loc] < score:
                    score = scores[loc]
                    new_process = [loc]
                elif scores[loc] == score:
                    new_process.append(loc)
        final_paths.extend(new_process)
        process = new_process

In [7]:
grid, max_rows, max_cols = {}, len(lines), len(lines[0])
for row, line in enumerate(lines):
    for col, value in enumerate(line):
        if value == 'E':
            end = (row, col)
        if value == 'S':
            start = (row, col)
        grid[(row, col)] = value

In [8]:
start_row, start_col = start
score, direction = 0, 0
process, score_dict = {(score, direction, start_row, start_col)}, {start: 0}
dir_dict = {}
while True:
    new_process = set()
    for path in process:
        score, direction, row, col = path
        for node in get_adjacent(grid, (row, col)):
            new_row, new_col = node
            new_direction = get_direction((row, col), node)
            diff = abs(new_direction - direction)
            rotations = min(diff, 4 - diff) 
            if rotations != 2:
                new_score = score + rotations * 1000 + 1
                if node not in score_dict or score_dict[node] > score:
                    new_process.add((new_score, new_direction, new_row, new_col))
                    score_dict[node] = new_score
    if new_process:
        process = new_process
    else:
        break
low_score = score_dict[end]
print(low_score)

11048


In [9]:
process, score_dict, path_dict =  [(0, 0, start, {start})], {start: 0}, {}
while True:
    new_process = []
    for path in process:
        score, direction, pos, node_set = path
        if pos == end:
            continue
        for loc in get_adjacent(grid, pos):
            if loc in node_set:
                continue
            new_direction = get_direction(pos, loc)
            diff = abs(new_direction - direction)
            rotations = min(diff, 4 - diff)
            new_score = score + rotations * 1000 + 1
            criteria_met = (rotations != 2 and new_score <= low_score
                            and (loc not in score_dict or new_score <= score_dict[loc]))
            if criteria_met:
                new_node_set = node_set.copy()
                new_node_set.add(loc)
                new_path = (new_score, new_direction, loc, new_node_set)
                new_process.append(new_path)
                if loc not in score_dict or new_score < score_dict[loc]:
                    score_dict[loc] = new_score
                    path_dict[loc] = new_node_set
                elif new_score == score_dict[loc]:
                    path_dict[loc].update(new_node_set)
    if new_process:
        process = new_process
    else:
        break
best_path = path_dict[end]
total_nodes = set()
for node in best_path:
    total_nodes.update((path_dict[node]))
print(len(total_nodes))

49


In [10]:
print_grid(grid, max_rows, max_cols, path_dict[end])

#################
#...#...#...#..O#
#.#.#.#.#.#.#.#O#
#.#.#.#...#...#O#
#.#.#.#.###.#.#O#
#OOO#.#.#.....#O#
#O#O#.#.#.#####O#
#O#O..#.#.#OOOOO#
#O#O#####.#O###.#
#O#O#..OOOOO#...#
#O#O###O#####.###
#O#O#OOO#.....#.#
#O#O#O#####.###.#
#O#O#O........#.#
#O#O#O#########.#
#O#OOO..........#
#################
